# SynthFace on Google Colab: 3D synthetic identities (approach A), one chunk per session

Renders synthetic people with **Blender 5.2.2 + MPFB 2.0.17** (same versions and assets as the dev laptop, every
download sha256-checked) on the Colab **GPU**, in **chunks** of about 150 people x 20 images (fits a ~5.5 h session).

**Every session:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`. Nothing else to set.
- The notebook finds the shared Drive folder, **resumes an unfinished chunk** or starts the next one
  (`p1_chunk01`, `p1_chunk02`, ...).
- **Uniqueness:** a new chunk's people are checked against **every earlier chunk** on Drive (face shape), so no
  person appears twice. Run **one session at a time** (two sessions at once could not see each other's picks).
- **Memory / disconnects:** a fresh Blender process per 25 people; renders go to the VM disk and are copied to
  Drive after each person. A time guard stops before the session ends; the next session continues the chunk.
- Keep the browser tab open (Colab stops idle sessions).

Rules (CLAUDE.md): license-clean assets only (all in `assets/LICENSES.csv`), no AI-generated faces, never commit `output/`.

## 1. Settings

In [ ]:
import time
SESSION_START = time.time()     # the time guard counts from here: run the notebook top to bottom

PHASE = "p1"                    # chunk names: p1_chunk01, p1_chunk02, ...
CHUNK = None                    # None = automatic (resume an unfinished chunk, else start the next one)
SEED = 20261013                 # keep the same seed for the whole project
# People per chunk, by group (blender/select_identities.py picks matching, non-duplicate identities):
MIX = "female=60,male_clean=30,male_stubble=15,male_short=15,male_full=15,male_goatee=15"   # = 150 people
IMAGES_PER_IDENTITY = 20
SAMPLES = 48                    # Cycles samples (+ denoiser)
DEVICE = "OPTIX"                # "OPTIX" (T4/L4/A100), "CUDA" if OptiX fails

SESSION_HOURS = 5.5             # Colab session length
SAFETY_MINUTES = 25             # start no new person this close to the end (copy to Drive + margin)
GROUP_SIZE = 25                 # people per Blender process (fresh process = memory reset)

DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1UhybtdRMuQroKlNt8RBiXOGVAkbDuIll"
DRIVE_DIR = None                # only if the folder is not found automatically, e.g. "/content/drive/MyDrive/SynthFace"

REPO = "https://github.com/ankitscode/-synthetic-Image-Genrator.git"   # public, no token needed
BRANCH = "master"
PROJECT = "/content/synthface"
TOOLS = "/content/synthface_tools"
BLENDER = f"{TOOLS}/blender-5.2.2-linux-x64/blender"
WORK = "/content/work"          # renders are written here (VM disk), then copied to Drive

## 2. Check the GPU and memory

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > T4 GPU"
!free -h | head -2; df -h /content | tail -1

## 3. Mount Drive, find the shared folder, get the code

The folder is found by its ID. If it is not found: open `DRIVE_FOLDER_URL` in Drive with **this** Google account,
right-click the folder > **Organize > Add shortcut > My Drive**, then run this cell again
(or set `DRIVE_DIR` in the settings).

In [ ]:
import glob, os, subprocess
from google.colab import drive

drive.mount("/content/drive")
FOLDER_ID = DRIVE_FOLDER_URL.rstrip("/").split("/")[-1].split("?")[0]


def find_drive_folder(folder_id):
    by_id = f"/content/drive/.shortcut-targets-by-id/{folder_id}"   # folders shared with you (via a shortcut)
    if os.path.isdir(by_id):
        subs = [os.path.join(by_id, d) for d in os.listdir(by_id) if os.path.isdir(os.path.join(by_id, d))]
        if len(subs) == 1:
            return subs[0]
    for root in ("/content/drive/MyDrive", "/content/drive/Shareddrives"):   # your own folder / a shared drive
        for path in [root, *glob.glob(f"{root}/*"), *glob.glob(f"{root}/*/*")]:
            try:
                if os.path.isdir(path) and os.getxattr(path, "user.drive.id").decode() == folder_id:
                    return path
            except OSError:
                pass
    return None


base = DRIVE_DIR or find_drive_folder(FOLDER_ID)
if not base:
    raise SystemExit("Shared folder not found: add a shortcut to it in My Drive (see above) or set DRIVE_DIR")
DRIVE_OUT = f"{base}/synthface_output"
os.makedirs(f"{DRIVE_OUT}/selections", exist_ok=True)
open(f"{DRIVE_OUT}/.write_test", "w").write("ok")      # fails here if this account cannot write
os.remove(f"{DRIVE_OUT}/.write_test")
print("Drive output:", DRIVE_OUT)

if not os.path.isdir(PROJECT):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, PROJECT], check=True)
else:
    subprocess.run(["git", "-C", PROJECT, "pull", "--ff-only"], check=True)
print("Code version:", subprocess.run(["git", "-C", PROJECT, "log", "-1", "--format=%h %cd %s", "--date=short"],
                                      capture_output=True, text=True).stdout)

## 4. Install Blender, MPFB, asset packs and HDRIs (about 3-5 min)

In [ ]:
!bash {PROJECT}/colab/setup_colab.sh {TOOLS}

## 5. Choose the chunk and its people

A new chunk: its people are picked and checked against every earlier selection on Drive (`synthface_output/selections/`,
about 5-10 min on the CPU). An unfinished chunk: its saved people and finished renders are reused.

In [ ]:
import json, re, shutil


def chunk_status(job_id):
    path = f"{DRIVE_OUT}/{job_id}/progress.json"
    return json.load(open(path))["status"] if os.path.exists(path) else None


existing = sorted(int(m.group(1)) for d in os.listdir(DRIVE_OUT) if (m := re.fullmatch(rf"{PHASE}_chunk(\d+)", d)))
lost = [f"{PHASE}_chunk{c:02d}" for c in existing
        if not os.path.exists(f"{DRIVE_OUT}/selections/{PHASE}_chunk{c:02d}.json")]
if lost:   # without them new chunks would pick the same people again
    raise SystemExit(f"Selections missing for {lost}: restore them in {DRIVE_OUT}/selections/ before rendering more")
existing += [int(m.group(1)) for f in os.listdir(f"{DRIVE_OUT}/selections")
             if (m := re.fullmatch(rf"{PHASE}_chunk(\d+)\.json", f)) and int(m.group(1)) not in existing]
if CHUNK is None:
    unfinished = sorted(c for c in existing if chunk_status(f"{PHASE}_chunk{c:02d}") != "rendered")
    CHUNK = unfinished[0] if unfinished else max(existing, default=0) + 1
JOB_ID = f"{PHASE}_chunk{CHUNK:02d}"
print("Chunk:", JOB_ID, "(resuming)" if CHUNK in existing else "(new)")

selection = f"{DRIVE_OUT}/selections/{JOB_ID}.json"
shapes = selection.replace(".json", ".shapes.npz")
if not os.path.exists(selection):
    earlier = sorted(glob.glob(f"{DRIVE_OUT}/selections/*.json"))
    local_sel = f"{WORK}/selections/{JOB_ID}.json"
    os.makedirs(os.path.dirname(local_sel), exist_ok=True)
    cmd = [BLENDER, "--background", "--python", "blender/select_identities.py", "--", "--seed", str(SEED),
           "--mix", MIX, "--out", local_sel] + (["--against", *earlier] if earlier else [])
    print(f"Picking people, checked against {len(earlier)} earlier selection(s) ...")
    r = subprocess.run(cmd, cwd=PROJECT, capture_output=True, text=True)
    print("\n".join(l for l in r.stdout.splitlines() if l.startswith(("AGAINST", "REJECT", "ACCEPTED", "SELECTION"))))
    if r.returncode != 0 or not os.path.exists(local_sel):
        raise RuntimeError("identity selection failed:\n" + r.stderr[-2000:])
    shutil.copyfile(local_sel.replace(".json", ".shapes.npz"), shapes)   # shapes first: the json marks "complete"
    shutil.copyfile(local_sel, selection)
sel = json.load(open(selection))
if sel["seed"] != SEED:
    raise SystemExit(f"{selection} uses seed {sel['seed']}, settings say {SEED}")
indices = sel["accepted"]
print(f"{len(indices)} people, candidates {sel['first_candidate']}-{sel['last_candidate']}, "
      f"near-duplicates rejected {len(sel['rejected_near_duplicates'])}, "
      f"compared with {sel['compared_with_earlier']} earlier people")

job_dir, drive_job = f"{WORK}/{JOB_ID}", f"{DRIVE_OUT}/{JOB_ID}"
if os.path.isdir(drive_job) and not os.path.isdir(job_dir):
    shutil.copytree(drive_job, job_dir, copy_function=shutil.copyfile)   # finished people of an earlier session
os.makedirs(job_dir, exist_ok=True)
job = {"job_id": JOB_ID, "seed": SEED, "num_identities": len(indices), "identity_indices": indices,
       "images_per_identity": IMAGES_PER_IDENTITY, "render": {"samples": SAMPLES, "device": DEVICE}}
job_path = f"{job_dir}/job.json"
if os.path.exists(job_path):
    if json.load(open(job_path)) != job:
        raise SystemExit(f"{JOB_ID} was started with other settings: set CHUNK to a new number or restore them")
else:
    json.dump(job, open(job_path, "w"), indent=2)

## 6. Render

One line per finished person; each person is copied to Drive as soon as it is done (a disconnect loses at most
the person in progress).
Stops by itself before the session ends; the next session resumes this chunk. Log: `<chunk>/blender_stderr.log`.

In [ ]:
def sync_to_drive(log=True):
    """Copy finished people + job files to Drive (each person once, after it is done)."""
    os.makedirs(drive_job, exist_ok=True)
    for name in os.listdir(job_dir):
        src, dst = f"{job_dir}/{name}", f"{drive_job}/{name}"
        if os.path.isfile(src) and (log or name != "blender_stderr.log"):
            shutil.copyfile(src, dst)
        elif name.startswith("id_") and not os.path.exists(f"{dst}/params.json"):
            params = f"{src}/params.json"
            if os.path.exists(params) and json.load(open(params)).get("status") == "done":
                shutil.copytree(src, dst, copy_function=shutil.copyfile, dirs_exist_ok=True,
                                ignore=shutil.ignore_patterns("params.json"))
                shutil.copyfile(params, f"{dst}/params.json")        # last: marks the person complete on Drive


stop_at = SESSION_START + SESSION_HOURS * 3600 - SAFETY_MINUTES * 60
done, t0, new, finished = set(), time.time(), 0, False
while not finished:
    if time.time() >= stop_at:
        break
    cmd = [BLENDER, "--background", "--python", "blender/render_batch.py", "--", "--job", job_path, "--resume",
           "--max-identities", str(GROUP_SIZE), "--stop-at", str(stop_at)]
    stopped = None
    with open(f"{job_dir}/blender_stderr.log", "a") as err:
        proc = subprocess.Popen(cmd, cwd=PROJECT, stdout=subprocess.PIPE, stderr=err, text=True)
        for line in proc.stdout:
            if not line.startswith('{"event"'):
                continue
            ev = json.loads(line)
            if ev["event"] == "stopped":
                stopped = ev
            elif ev["status"] == "done":
                if ev["index"] not in done and not ev.get("skipped"):
                    new += 1
                    print(f"id_{ev['index']:05d} ({len(done) + 1}/{len(indices)})  {time.time() - t0:.0f}s, "
                          f"{(time.time() - t0) / new:.0f}s per person", flush=True)
                done.add(ev["index"])
                if not ev.get("skipped"):
                    sync_to_drive(log=False)   # each person to Drive as soon as it is done
            else:
                print(f"id_{ev['index']:05d} FAILED (see blender_stderr.log; retried next run)", flush=True)
        code = proc.wait()
    sync_to_drive()
    if code != 0:
        raise RuntimeError(f"Blender exit code {code}: see {drive_job}/blender_stderr.log")
    finished = stopped is None
    if stopped and stopped["reason"] == "time_limit":
        break
status = chunk_status(JOB_ID)
print(f"{JOB_ID}: {len(done)}/{len(indices)} people done, {new} this session, status {status}")
if status != "rendered":
    print("Session time is nearly up: open a new session and Run all; it continues this chunk.")

## 7. Check the chunk (no Blender, about 1-2 min)

Aligns every image to 112x112 from the 3D landmarks, then checks: all 5 landmarks visible, same person in every image,
no near-duplicate people (face shapes from the selection). Results go to Drive with the chunk.

In [ ]:
!cd {PROJECT} && python -m app.postprocess {job_dir} 2>&1 | tail -3
!cd {PROJECT} && python tools/validate_job.py {job_dir} --shapes "{shapes}"
!cd {PROJECT} && python tools/contact_sheet.py {job_dir} --max-cols 5 > /dev/null
for name in ("validation.json", "validation_pairs.png", "contact_sheet.png"):
    shutil.copyfile(f"{job_dir}/{name}", f"{drive_job}/{name}")
for d in glob.glob(f"{job_dir}/id_*"):        # aligned crops + alignment.json of every person
    if os.path.isdir(f"{d}/aligned"):
        dst = f"{drive_job}/{os.path.basename(d)}"
        shutil.copytree(f"{d}/aligned", f"{dst}/aligned", copy_function=shutil.copyfile, dirs_exist_ok=True)
        shutil.copyfile(f"{d}/alignment.json", f"{dst}/alignment.json")

from PIL import Image as PILImage
from IPython.display import display
sheet = PILImage.open(f"{job_dir}/contact_sheet.png")
display(sheet.crop((0, 0, sheet.width, min(sheet.height, 2600))))   # first ~20 people (the full sheet is on Drive)

## 8. Next

- Next chunk: a **new session** (or this one, if more than ~4.5 h are left) > `Run all`.
- Bring results to the laptop: download `synthface_output/<chunk>/` to `output/<chunk>/` and
  `synthface_output/selections/` to `output/selections/` in the project.